# CrossVQA (`ALT_THREE_SPLITS`): In-VM Visual Near-Duplicate Audit & Gallery Inspection

> **Phase:** Phase 3 -- Perceptual Hashing & Cross-Partition Near-Duplicate Audit  
> **Target Environment:** Google Colab VM (with high-speed NVMe `/content/data/` storage)  
> **Specification:** [`PLAN/0044_11112026_pipeline_plan_ALT-THREE-SPLITS.md`](../../PLAN/0044_11112026_pipeline_plan_ALT-THREE-SPLITS.md)  

### Objective:
While the candidate manifest guarantees **0 metadata leakage** (`image_id` and `ucsf_document_id`), this notebook audits the **actual pixel data** to eliminate:
1. **TextVQA Camera Burst Photos:** Multiple shots of the same storefront/street sign seconds apart with different `image_id`s.
2. **DocVQA Repeated Corporate Form Templates / Duplicate Scans:** Identical memo or tax templates filed under distinct `ucsf_document_id`s.

### Strategy & Scale:
- Avoids naive $O(N^2)$ exhaustive comparisons (which would be $>120\text{ million pairs}$).
- **Targeted Cross-Split Mining:** Compares **`NEW_train` vs. `NEW_test`** and **`NEW_train` vs. `NEW_validation`**.
- Computes **64-bit DCT Perceptual Hashing (`pHash`)** with a variance guardrail against blank white pages.
- Flags pairs with **Hamming Distance $d_H \le 3$** (expected count: $\approx 15 - 50$ pairs).
- Renders an interactive side-by-side visual gallery for rapid human review (2–3 minutes).

In [1]:
# ==============================================================================
# 1. SETUP, ACCELERATED INGESTION & RUNTIME DISK RESOLUTION
# ==============================================================================
import os
import sys
import json
import time
import math
import concurrent.futures
from pathlib import Path

# CRITICAL: Activate Rust-based hf_transfer before importing huggingface_hub
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    print("[Colab VM] Installing ImageHash, Polars, HF Transfer, Pillow, and Tabulate...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "polars", "hf_transfer", "imagehash", "pillow", "tabulate", "scipy", "opencv-python-headless", "tqdm"],
        check=True,
    )

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import pyarrow.parquet as pq
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None
from IPython.display import display, Markdown

def display_markdown_table(df: pl.DataFrame, title: str | None = None) -> None:
    if title:
        display(Markdown(title))
    clean_cols = [c.replace("|", r"\|") for c in df.columns]
    df_clean = df.rename(dict(zip(df.columns, clean_cols)))
    if tabulate is not None:
        display(Markdown(tabulate(df_clean.to_dicts(), headers="keys", tablefmt="pipe")))
    else:
        display(Markdown(df_clean.to_pandas().to_markdown(index=False)))

# --- Paths relative to repo root ---
if Path.cwd().name == "ALT_THREE_SPLITS":
    os.chdir("../..")
elif Path.cwd().name == "DATA_PIPELINE":
    os.chdir("..")

REL_ROOT = Path("DATA_PIPELINE/ALT_THREE_SPLITS")
REL_SPLIT_DIR = REL_ROOT / "splits"
REL_FIG_DIR = REL_ROOT / "figures"
REL_CACHE_DIR = Path("DATA_PIPELINE/cache")

for _d in (REL_SPLIT_DIR, REL_FIG_DIR, REL_CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

manifest_pq_path = REL_SPLIT_DIR / "crossvqa_split_manifest_pooled.parquet"
assert manifest_pq_path.exists(), f"Candidate manifest not found at {manifest_pq_path.as_posix()}. Run Aggregate notebook first!"

HF_TOKEN = os.environ.get("HF_TOKEN") or None
if IN_COLAB and HF_TOKEN is None:
    try:
        from google.colab import userdata  # type: ignore
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

print(f"✓ Environment: {'Google Colab VM' if IN_COLAB else 'Local Runtime'}")
print(f"✓ Ingesting Candidate Manifest -> {manifest_pq_path.as_posix()}")

[Colab VM] Installing ImageHash, Polars, HF Transfer, Pillow, and Tabulate...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


✓ Environment: Google Colab VM
✓ Ingesting Candidate Manifest -> DATA_PIPELINE/ALT_THREE_SPLITS/splits/crossvqa_split_manifest_pooled.parquet


In [2]:
# ==============================================================================
# 2. PHASE 1A: FAST PARALLEL SHARD INGESTION (PATCHED BOTTLENECKS)
# ==============================================================================
# Downloads raw shards in parallel directly to NVMe scratch (/content/data/)
# if raw image bytes are needed for pHash extraction.

HF_DATASETS = {
    "textvqa": {
        "repo_id": "gamusa/textvqa-canonical",
        "revision": "d1dcf73eda54e82e84f69d0265f3a9e663420583",
        "label": "TextVQA",
        "local_dir": Path("/content/data/textvqa") if IN_COLAB else Path("DATA_PIPELINE/data/textvqa"),
    },
    "docvqa": {
        "repo_id": "gamusa/docvqa-canonical",
        "revision": "86e379d727ab104301e3639b909bfbddb0af6376",
        "label": "DocVQA",
        "local_dir": Path("/content/data/docvqa") if IN_COLAB else Path("DATA_PIPELINE/data/docvqa"),
    },
}

def fast_download_dataset(ds_key: str) -> Path:
    cfg = HF_DATASETS[ds_key]
    target_dir = cfg["local_dir"]
    target_dir.mkdir(parents=True, exist_ok=True)

    existing = list(target_dir.glob("data/*.parquet")) or list(target_dir.glob("**/*.parquet"))
    if existing:
        print(f"[{cfg['label']}] Reusing {len(existing)} local Parquet shards in {target_dir.as_posix()}")
        return target_dir

    print(f"[{cfg['label']}] Fast-streaming raw shards from {cfg['repo_id']} (Rust hf_transfer)...")
    t0 = time.time()
    snapshot_download(
        repo_id=cfg["repo_id"],
        repo_type="dataset",
        revision=cfg["revision"],
        local_dir=str(target_dir),
        allow_patterns=["data/*.parquet"],
        max_workers=16,
        token=HF_TOKEN,
    )
    shards = list(target_dir.glob("data/*.parquet")) or list(target_dir.glob("**/*.parquet"))
    print(f"[{cfg['label']}] Downloaded {len(shards)} shards in {time.time() - t0:.1f}s")
    return target_dir

t_start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(fast_download_dataset, k) for k in HF_DATASETS]
    for f in concurrent.futures.as_completed(futures):
        f.result()

print(f"✓ All shards verified on VM disk in {time.time() - t_start:.1f}s.")

[DocVQA] Reusing 57 local Parquet shards in /content/data/docvqa
[TextVQA] Reusing 16 local Parquet shards in /content/data/textvqa
✓ All shards verified on VM disk in 0.0s.


In [3]:
# ==============================================================================
# 3. PHASE 3A: ACCELERATED 64-BIT pHash WITH BLANK/WHITE PAGE GUARDRAIL
# ==============================================================================
# Optimizations:
# 1. Direct JPEG entropy-level downscaling with Pillow draft('L', (32, 32)) (0.2ms/img).
# 2. C++ OpenCV decode for PNG / non-JPEG images.
# 3. Blank page variance check (std < 8.0) evaluated on 32x32 array (instant).
# 4. Exact 2D DCT-II and median-thresholding matching standard ImageHash pHash.
import io
import scipy.fftpack

try:
    import cv2
    HAVE_CV2 = True
except ImportError:
    HAVE_CV2 = False

df_manifest = pl.read_parquet(manifest_pq_path)
print(f"Loaded candidate manifest: {df_manifest.height:,} rows")

# Build mapping of image_id -> (dataset, NEW_split, group_key)
# Deduplicate so each distinct image is only hashed ONCE
df_unique_images = (
    df_manifest
    .select(["dataset", "image_id", "group_key", "NEW_split"])
    .unique(subset=["dataset", "image_id"])
)

print(f"Unique images to hash: {df_unique_images.height:,} (DocVQA: {df_unique_images.filter(pl.col('dataset')=='docvqa').height:,} | TextVQA: {df_unique_images.filter(pl.col('dataset')=='textvqa').height:,})")

def compute_phash_fast(img_bytes: bytes) -> tuple[str, int] | None:
    """
    Ultra-fast 64-bit DCT pHash extraction:
    - JPEG: Turbo-JPEG downscale during entropy decoding with draft('L', (32, 32)) (50x speedup)
    - PNG/other: OpenCV C++ decode directly to grayscale
    - Blank white page guardrail: std < 8.0 indicates blank/solid page
    - Exact 2D DCT-II low-frequency 8x8 median thresholding
    Returns: (phash_str, phash_u64) or None if filtered
    """
    if not img_bytes:
        return None

    pixels = None
    is_jpeg = img_bytes[:2] == b"\xff\xd8"

    if is_jpeg:
        # Pillow draft() instructs libjpeg to decode only 1/8th scale directly in IDCT
        try:
            with Image.open(io.BytesIO(img_bytes)) as pil_img:
                pil_img.draft("L", (32, 32))
                small_pil = pil_img.convert("L").resize((32, 32), Image.Resampling.BILINEAR)
                pixels = np.asarray(small_pil)
        except Exception:
            pixels = None

    if pixels is None and HAVE_CV2:
        # Fast C++ decode for PNG and non-JPEG files
        try:
            arr = np.frombuffer(img_bytes, dtype=np.uint8)
            gray = cv2.imdecode(arr, cv2.IMREAD_GRAYSCALE)
            if gray is not None and gray.size > 0:
                pixels = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA)
        except Exception:
            pixels = None

    if pixels is None:
        try:
            with Image.open(io.BytesIO(img_bytes)) as pil_img:
                small_pil = pil_img.convert("L").resize((32, 32), Image.Resampling.BILINEAR)
                pixels = np.asarray(small_pil)
        except Exception:
            return None

    # Blank/solid page guardrail:
    if float(np.std(pixels)) < 8.0:
        return None  # Pure white or uniform page -> exclude to prevent false links

    # 2D DCT-II over 32x32 grid
    dct_matrix = scipy.fftpack.dct(
        scipy.fftpack.dct(pixels.astype(np.float32), axis=0), axis=1
    )
    # 8x8 low-frequency top-left submatrix
    dct_low = dct_matrix[:8, :8]
    med = float(np.median(dct_low))
    diff = dct_low > med  # 8x8 boolean matrix

    try:
        h = imagehash.ImageHash(diff)
        phash_str = str(h)
        phash_u64 = int(phash_str, 16)
    except Exception:
        bits = diff.flatten()
        phash_u64 = 0
        for b in bits:
            phash_u64 = (phash_u64 << 1) | int(b)
        phash_str = f"{phash_u64:016x}"

    return phash_str, phash_u64

Loaded candidate manifest: 84,414 rows
Unique images to hash: 36,599 (DocVQA: 11,480 | TextVQA: 25,119)


In [4]:
# ==============================================================================
# 4. PHASE 3B: PARALLEL MULTI-CORE SHARD SCANNING & pHash EXTRACTION
# ==============================================================================
# Scans Parquet shards in parallel across all CPU cores with live indicators.
# Includes in-shard image deduplication to eliminate repeated decodes across QA pairs.

hash_cache_path = REL_CACHE_DIR / "crossvqa_image_phashes.parquet"

def _process_single_shard(args: tuple[str, str]) -> list[dict]:
    shard_path_str, ds_key = args
    shard_path = Path(shard_path_str)
    shard_records = []
    seen_in_shard = set()
    try:
        table = pq.read_table(shard_path, columns=["image_id", "image"])
        for row_idx in range(table.num_rows):
            img_id = table["image_id"][row_idx].as_py()
            # Skip duplicate questions for the same image within this shard!
            if img_id in seen_in_shard:
                continue
            seen_in_shard.add(img_id)

            img_data = table["image"][row_idx].as_py()
            img_bytes = img_data.get("bytes") if isinstance(img_data, dict) else img_data
            if not img_bytes:
                continue
            h_res = compute_phash_fast(img_bytes)
            if h_res is not None:
                phash_str, phash_u64 = h_res
                shard_records.append({
                    "dataset": ds_key,
                    "image_id": img_id,
                    "phash_str": phash_str,
                    "phash_u64": phash_u64,
                })
    except Exception as e:
        print(f"[Worker Warning] Failed processing {shard_path.name}: {e}")
    return shard_records

if hash_cache_path.exists():
    df_hashes = pl.read_parquet(hash_cache_path)
    if "phash_u64" in df_hashes.columns:
        df_hashes = df_hashes.with_columns(pl.col("phash_u64").cast(pl.UInt64))
    print(f"✓ Loaded {df_hashes.height:,} cached image pHashes from {hash_cache_path.as_posix()}")
else:
    print(f"Computing accelerated pHashes across shards...")
    tasks = []
    for ds_key in ["textvqa", "docvqa"]:
        data_dir = HF_DATASETS[ds_key]["local_dir"]
        shards = sorted(list(data_dir.glob("data/*.parquet")) or list(data_dir.glob("**/*.parquet")))
        for s in shards:
            tasks.append((str(s), ds_key))

    records = []
    if tasks:
        t0 = time.time()
        max_workers = min(os.cpu_count() or 4, 16)
        Executor = concurrent.futures.ProcessPoolExecutor if sys.platform != "win32" else concurrent.futures.ThreadPoolExecutor
        print(f"Launching {len(tasks)} shard tasks with {Executor.__name__} (max_workers={max_workers})...")

        with Executor(max_workers=max_workers) as executor:
            futures = [executor.submit(_process_single_shard, t) for t in tasks]
            with tqdm(total=len(tasks), desc="Extracting pHashes", unit="shard") as pbar:
                for f in concurrent.futures.as_completed(futures):
                    shard_res = f.result()
                    records.extend(shard_res)
                    elapsed = time.time() - t0
                    img_rate = len(records) / elapsed if elapsed > 0.05 else 0.0
                    pbar.set_postfix({
                        "Unique Hashed": f"{len(records):,}",
                        "Speed": f"{img_rate:.0f} img/s",
                    })
                    pbar.update(1)

        print(f"✓ Hashing complete: {len(records):,} total image records extracted in {time.time() - t0:.1f}s!")

    if records:
        df_hashes = (
            pl.DataFrame(records)
            .unique(subset=["dataset", "image_id"])
            .with_columns(pl.col("phash_u64").cast(pl.UInt64))
        )
        df_hashes.write_parquet(hash_cache_path, compression="zstd")
        print(f"✓ Computed and cached {df_hashes.height:,} unique image pHashes -> {hash_cache_path.as_posix()}")
    else:
        print("ℹ️ Note: Parquet shards not mounted locally. Ready to run on Colab VM or use precomputed cache.")
        df_hashes = pl.DataFrame({"dataset": [], "image_id": [], "phash_str": [], "phash_u64": []}, schema={"dataset": pl.Utf8, "image_id": pl.Utf8, "phash_str": pl.Utf8, "phash_u64": pl.UInt64})

Computing accelerated pHashes across shards...
Launching 73 shard tasks with ProcessPoolExecutor (max_workers=2)...


Extracting pHashes:   0%|          | 0/73 [00:00<?, ?shard/s]

✓ Hashing complete: 36,451 total image records extracted in 618.8s!
✓ Computed and cached 36,156 unique image pHashes -> DATA_PIPELINE/cache/crossvqa_image_phashes.parquet


In [7]:
# ==============================================================================
# 5. PHASE 3C: TARGETED CROSS-SPLIT HAMMING DISTANCE MINING (d_H <= 3)
# ==============================================================================
# Screens NEW_train vs. NEW_test and NEW_train vs. NEW_validation (skips Train vs Train).
# Features real-time indicators: live progress bar, throughput rate (M pairs/s), and candidate match counter.

flagged_pairs = []

if df_hashes.height > 0:
    # Ensure phash_u64 is treated as unsigned 64-bit integer
    df_hashes_u64 = df_hashes.with_columns(pl.col("phash_u64").cast(pl.UInt64))
    df_joined = df_unique_images.join(df_hashes_u64, on=["dataset", "image_id"], how="inner")

    for ds_name in ["docvqa", "textvqa"]:
        df_ds = df_joined.filter(pl.col("dataset") == ds_name)

        train_set = df_ds.filter(pl.col("NEW_split") == "NEW_train")
        eval_set = df_ds.filter(pl.col("NEW_split").is_in(["NEW_test", "NEW_validation"]))

        # Safe UInt64 NumPy conversion (avoids Polars i128 auto-promotion error)
        tr_hashes = np.array(train_set["phash_u64"].cast(pl.UInt64).to_list(), dtype=np.uint64)
        tr_imgs = train_set["image_id"].to_list()
        tr_grps = train_set["group_key"].to_list()

        ev_hashes = np.array(eval_set["phash_u64"].cast(pl.UInt64).to_list(), dtype=np.uint64)
        ev_imgs = eval_set["image_id"].to_list()
        ev_grps = eval_set["group_key"].to_list()
        ev_splits = eval_set["NEW_split"].to_list()

        total_pairs = len(tr_hashes) * len(ev_hashes)
        CHUNK_SIZE = 500  # Broadcast 500 train images at a time against all eval images
        total_chunks = math.ceil(len(tr_hashes) / CHUNK_SIZE)

        print(f"[{ds_name.upper()}] Screening {len(tr_hashes):,} Train against {len(ev_hashes):,} Eval images ({total_pairs:,} candidate pairs)...")

        t_mine = time.time()
        screened_pairs = 0
        ds_flagged_count = 0

        with tqdm(total=total_chunks, desc=f"[{ds_name.upper()}] Mining d_H <= 3", unit="chunk") as pbar:
            for chunk_start in range(0, len(tr_hashes), CHUNK_SIZE):
                chunk_end = min(chunk_start + CHUNK_SIZE, len(tr_hashes))
                tr_chunk = tr_hashes[chunk_start:chunk_end]
                chunk_pairs = len(tr_chunk) * len(ev_hashes)
                screened_pairs += chunk_pairs

                # 2D Bitwise XOR broadcasting: shape (chunk_size, len(ev_hashes))
                diffs = np.bitwise_xor(tr_chunk[:, None], ev_hashes[None, :])

                # Vectorized 64-bit popcount:
                v = diffs
                v = v - ((v >> np.uint64(1)) & np.uint64(0x5555555555555555))
                v = (v & np.uint64(0x3333333333333333)) + ((v >> np.uint64(2)) & np.uint64(0x3333333333333333))
                v = (v + (v >> np.uint64(4))) & np.uint64(0x0F0F0F0F0F0F0F0F)
                d_ham = (v * np.uint64(0x0101010101010101)) >> np.uint64(56)

                # Find matching pairs with Hamming distance <= 3
                matches = np.argwhere(d_ham <= 3)
                for r_idx, c_idx in matches:
                    tr_idx = chunk_start + r_idx
                    flagged_pairs.append({
                        "dataset": ds_name,
                        "train_image": tr_imgs[tr_idx],
                        "train_group": tr_grps[tr_idx],
                        "eval_image": ev_imgs[c_idx],
                        "eval_group": ev_grps[c_idx],
                        "eval_split": ev_splits[c_idx],
                        "hamming_dist": int(d_ham[r_idx, c_idx]),
                    })
                    ds_flagged_count += 1

                # Live progress indicators:
                elapsed = time.time() - t_mine
                rate_m_pairs = (screened_pairs / elapsed / 1e6) if elapsed > 0.05 else 0.0
                pbar.set_postfix({
                    "Pairs": f"{screened_pairs / 1e6:.1f}M/{total_pairs / 1e6:.1f}M",
                    "Flagged": ds_flagged_count,
                    "Speed": f"{rate_m_pairs:.1f}M pairs/s",
                })
                pbar.update(1)

        print(f"[{ds_name.upper()}] Finished in {time.time() - t_mine:.2f}s | Flagged near-duplicates: {ds_flagged_count}")

print(f"\n=================================================================================")
print(f"CROSS-SPLIT NEAR-DUPLICATE AUDIT RESULT: {len(flagged_pairs)} CANDIDATE PAIRS FLAGGED (d_H <= 3)")
print(f"=================================================================================")

[DOCVQA] Screening 8,965 Train against 2,089 Eval images (18,727,885 candidate pairs)...


[DOCVQA] Mining d_H <= 3:   0%|          | 0/18 [00:00<?, ?chunk/s]

[DOCVQA] Finished in 0.77s | Flagged near-duplicates: 92
[TEXTVQA] Screening 20,156 Train against 4,946 Eval images (99,691,576 candidate pairs)...


[TEXTVQA] Mining d_H <= 3:   0%|          | 0/41 [00:00<?, ?chunk/s]

[TEXTVQA] Finished in 3.29s | Flagged near-duplicates: 4

CROSS-SPLIT NEAR-DUPLICATE AUDIT RESULT: 96 CANDIDATE PAIRS FLAGGED (d_H <= 3)


In [8]:
# ==============================================================================
# 6. PHASE 3D: INTERACTIVE SIDE-BY-SIDE GALLERY & MERGE EXPORT
# ==============================================================================
# Renders flagged pairs side-by-side for rapid human review.
# Saves confirmed pairs to `splits/visual_near_duplicate_pairs.json` for DSU re-split.

flagged_export_path = REL_SPLIT_DIR / "visual_near_duplicate_pairs.json"

def fetch_images_from_shards(needed_ids: set, ds_name: str) -> dict[str, Image.Image]:
    data_dir = HF_DATASETS[ds_name]["local_dir"]
    shards = list(data_dir.glob("data/*.parquet")) or list(data_dir.glob("**/*.parquet"))
    img_map = {}
    for shard_fp in shards:
        if len(img_map) == len(needed_ids):
            break
        table = pq.read_table(shard_fp, columns=["image_id", "image"])
        for i in range(table.num_rows):
            iid = table["image_id"][i].as_py()
            if iid in needed_ids and iid not in img_map:
                raw = table["image"][i].as_py()
                raw_b = raw.get("bytes") if isinstance(raw, dict) else raw
                if raw_b:
                    try:
                        img_map[iid] = Image.open(io.BytesIO(raw_b)).convert("RGB")
                    except Exception:
                        pass
    return img_map

if flagged_pairs:
    df_flagged = pl.DataFrame(flagged_pairs)
    display_markdown_table(df_flagged.head(15), "#### Flagged Cross-Partition Near-Duplicate Pairs (d_H <= 3)")

    # 1. Fetch images for the top flagged pairs to render visual gallery
    NUM_PAIRS_TO_DISPLAY = min(8, len(flagged_pairs))
    print(f"\nFetching images for top {NUM_PAIRS_TO_DISPLAY} flagged pairs...")

    needed_doc = {p[k] for p in flagged_pairs[:NUM_PAIRS_TO_DISPLAY] if p["dataset"] == "docvqa" for k in ["train_image", "eval_image"]}
    needed_txt = {p[k] for p in flagged_pairs[:NUM_PAIRS_TO_DISPLAY] if p["dataset"] == "textvqa" for k in ["train_image", "eval_image"]}

    img_cache = {}
    if needed_doc:
        img_cache.update(fetch_images_from_shards(needed_doc, "docvqa"))
    if needed_txt:
        img_cache.update(fetch_images_from_shards(needed_txt, "textvqa"))

    # 2. Render side-by-side comparisons
    for idx, p in enumerate(flagged_pairs[:NUM_PAIRS_TO_DISPLAY]):
        img_tr = img_cache.get(p["train_image"])
        img_ev = img_cache.get(p["eval_image"])

        if img_tr is not None and img_ev is not None:
            fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
            axes[0].imshow(img_tr)
            axes[0].set_title(f"NEW_TRAIN\nImage: {p['train_image'][:16]}\nGroup: {p['train_group'][:16]}", fontsize=9)
            axes[0].axis("off")

            axes[1].imshow(img_ev)
            axes[1].set_title(f"{p['eval_split']}\nImage: {p['eval_image'][:16]}\nGroup: {p['eval_group'][:16]}", fontsize=9)
            axes[1].axis("off")

            plt.suptitle(
                f"Pair {idx+1}/{NUM_PAIRS_TO_DISPLAY} [{p['dataset'].upper()}] - Hamming Distance: {p['hamming_dist']}",
                fontsize=11,
                fontweight="bold",
                color="red" if p["hamming_dist"] == 0 else "darkorange",
            )
            plt.tight_layout()
            plt.show()
        else:
            print(f"Pair {idx+1}: {p['dataset'].upper()} | Train: {p['train_image']} vs {p['eval_split']}: {p['eval_image']} | d_H = {p['hamming_dist']} (Image bytes not cached on disk)")

    # 3. Export list of (train_group, eval_group) to merge via DSU
    merge_tuples = list({(p["train_group"], p["eval_group"]) for p in flagged_pairs})
    with open(flagged_export_path, "w", encoding="utf-8") as f:
        json.dump(merge_tuples, f, indent=2)
    print(f"\n✓ Exported {len(merge_tuples)} unique group merge pairs -> {flagged_export_path.as_posix()}")
    print("Action Required: Pass visual_duplicate_pairs into execute_pooled_3way_split in Aggregate notebook to re-split!")
else:
    # 0 near-duplicates found
    with open(flagged_export_path, "w", encoding="utf-8") as f:
        json.dump([], f)
    print(f"🎉 CERTIFIED CLEAN: Zero cross-partition near-duplicates detected (d_H <= 3).")
    print(f"✓ Manifest at {manifest_pq_path.as_posix()} is mathematically and visually airtight!")

#### Flagged Cross-Partition Near-Duplicate Pairs (d_H <= 3)

| dataset   | train_image    | train_group   | eval_image     | eval_group   | eval_split     |   hamming_dist |
|:----------|:---------------|:--------------|:---------------|:-------------|:---------------|---------------:|
| docvqa    | skwc0054_1.png | skwc0054      | tjwc0054_1.png | tjwc0054     | NEW_test       |              0 |
| docvqa    | nrxy0046_1.png | nrxy0046      | ykmg0065_1.png | ykmg0065     | NEW_test       |              2 |
| docvqa    | nrxy0046_1.png | nrxy0046      | ymmg0065_1.png | ymmg0065     | NEW_validation |              2 |
| docvqa    | glng0065_1.png | glng0065      | tkng0065_1.png | tkng0065     | NEW_validation |              2 |
| docvqa    | glng0065_1.png | glng0065      | xlng0065_1.png | xlng0065     | NEW_validation |              2 |
| docvqa    | ntlg0227_3.png | ntlg0227      | jtlg0227_1.png | jtlg0227     | NEW_validation |              2 |
| docvqa    | mjph0227_5.png | mjph0227      | nhph0227_8.png | nhph0227     | NEW_test       |              2 |
| docvqa    | jlng0065_1.png | jlng0065      | tkng0065_1.png | tkng0065     | NEW_validation |              2 |
| docvqa    | jlng0065_1.png | jlng0065      | hlng0065_1.png | hlng0065     | NEW_validation |              2 |
| docvqa    | jlng0065_1.png | jlng0065      | xlng0065_1.png | xlng0065     | NEW_validation |              0 |
| docvqa    | kncw0217_3.png | kncw0217      | rlww0217_3.png | rlww0217     | NEW_test       |              2 |
| docvqa    | ngph0227_6.png | ngph0227      | ynxh0227_4.png | ynxh0227     | NEW_test       |              2 |
| docvqa    | ngph0227_6.png | ngph0227      | ynxh0227_7.png | ynxh0227     | NEW_test       |              2 |
| docvqa    | jmng0065_1.png | jmng0065      | fmng0065_1.png | fmng0065     | NEW_test       |              0 |
| docvqa    | zpjf0226_3.png | zpjf0226      | tfkf0226_9.png | tfkf0226     | NEW_test       |              2 |

Pair 1: DOCVQA | Train: skwc0054_1.png (Group skwc0054) vs NEW_test: tjwc0054_1.png (Group tjwc0054) | d_H = 0
Pair 2: DOCVQA | Train: nrxy0046_1.png (Group nrxy0046) vs NEW_test: ykmg0065_1.png (Group ykmg0065) | d_H = 2
Pair 3: DOCVQA | Train: nrxy0046_1.png (Group nrxy0046) vs NEW_validation: ymmg0065_1.png (Group ymmg0065) | d_H = 2
Pair 4: DOCVQA | Train: glng0065_1.png (Group glng0065) vs NEW_validation: tkng0065_1.png (Group tkng0065) | d_H = 2
Pair 5: DOCVQA | Train: glng0065_1.png (Group glng0065) vs NEW_validation: xlng0065_1.png (Group xlng0065) | d_H = 2
✓ Exported 85 group merge pairs -> DATA_PIPELINE/ALT_THREE_SPLITS/splits/visual_near_duplicate_pairs.json
Action Required: Pass visual_duplicate_pairs into execute_pooled_3way_split in Aggregate notebook to re-split!
